# Personal AI Avatar Video Pipeline

**Voice Cloning (XTTS v2) + Talking Head (SadTalker)**

---

This notebook generates a talking-head video of **you** speaking any text script, using only:
- A short reference video of yourself speaking (~1 minute)
- The text you want to say

### How it works
1. **XTTS v2** (Coqui) clones your voice from the reference audio and synthesizes new speech
2. **SadTalker** animates a photo of your face to match the synthesized audio (lip sync)
3. **GFPGAN** enhances the face quality in the output video

---

## Section 0: Prerequisites & Requirements

### GPU Requirement
> **IMPORTANT**: This notebook requires a GPU runtime. Go to **Runtime → Change runtime type → GPU** before running.
> - Recommended: T4 (free tier), A100 (faster, Colab Pro)
> - Minimum VRAM: 6 GB
> - SadTalker inference for a 1-minute video takes ~3–5 minutes on T4

### Reference Video Requirements
Record a reference video of yourself with these guidelines for best results:

| Requirement | Details |
|---|---|
| **Duration** | 30 seconds – 2 minutes (longer = better voice clone) |
| **Face visibility** | Front-facing, full face visible, no sunglasses |
| **Lighting** | Even, natural or soft artificial light. Avoid harsh shadows |
| **Background noise** | Quiet environment. No music, TV, or echo |
| **Speech** | Speak naturally and clearly. Read a passage or talk freely |
| **Camera** | Eye-level, stable. Phone propped up or on tripod |
| **Resolution** | At least 720p |
| **Format** | MP4, MOV, or AVI |

### Estimated Runtime on T4
- Setup & installs: ~5–8 minutes (first run only)
- Model downloads: ~3–5 minutes (first run only)
- XTTS voice synthesis: ~1–2 minutes per minute of speech
- SadTalker inference: ~3–5 minutes per minute of video

---

**All models are 100% open source. No API keys required.**

---
## Section 1: Environment Setup

Run these cells in order. Installation takes ~5–8 minutes on first run.

In [ ]:
# Cell 1.1 — Check GPU availability
import subprocess
import sys

print("Checking GPU...")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected! Please enable GPU runtime:\n"
        "Runtime → Change runtime type → Hardware accelerator → GPU → Save"
    )

print(result.stdout)
print("GPU detected. Ready to proceed.")

# Also check via PyTorch
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nUsing device: {device}")
except ImportError:
    print("PyTorch not yet installed (will be installed with TTS).")
    device = "cuda"

In [ ]:
# Cell 1.2 — Pin torch ecosystem FIRST
# Chatterbox TTS requires torch 2.6.0 + torchaudio 2.6.0.
# This must run before any other install that might pull a newer torch.
# After this cell finishes: Runtime -> Restart session, then run all remaining cells.

print('Pinning torch 2.6.0 + torchaudio 2.6.0 (CUDA 12.6)...')
print('This takes ~2 minutes. Do NOT skip this step.')

!pip install \
    torch==2.6.0 \
    torchaudio==2.6.0 \
    torchvision==0.21.0 \
    --index-url https://download.pytorch.org/whl/cu126 \
    --force-reinstall -q

print('')
print('='*55)
print('DONE. Now go to: Runtime -> Restart session')
print('Then run all cells from Cell 1.1 downward.')
print('='*55)


In [ ]:
# Cell 1.2 — Install system packages
# ffmpeg: audio/video processing
# libsndfile1: audio file I/O (required by TTS)
# libgl1-mesa-glx: OpenGL support (required by OpenCV in SadTalker)
# cmake: required for building some Python packages

print("Installing system packages...")
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1 libgl1-mesa-glx cmake
print("System packages installed.")

# Verify ffmpeg
!ffmpeg -version 2>&1 | head -1

In [ ]:
# Cell 1.3 — Clone SadTalker and install its dependencies
# SadTalker: https://github.com/OpenTalker/SadTalker
# Animates a static face image to match driving audio (lip sync)

import os

SADTALKER_DIR = "/content/SadTalker"

if os.path.exists(SADTALKER_DIR):
    print(f"SadTalker already cloned at {SADTALKER_DIR}. Skipping clone.")
else:
    print("Cloning SadTalker...")
    !git clone https://github.com/OpenTalker/SadTalker.git {SADTALKER_DIR}
    print("SadTalker cloned.")

print("Installing SadTalker requirements...")
!pip install -q -r {SADTALKER_DIR}/requirements.txt
print("SadTalker requirements installed.")

In [ ]:
# Cell 1.4 — Install Chatterbox TTS
# torch 2.6.0 is already pinned from Cell 1.2 so this installs cleanly.

!pip install chatterbox-tts -q

from chatterbox.tts import ChatterboxTTS
print('Chatterbox TTS ready.')


---
## Section 2: Download Model Checkpoints

This section downloads the pre-trained model weights. Downloads happen once and are cached.

| Model | Size | Purpose |
|---|---|---|
| SadTalker checkpoints | ~600 MB | Face animation & lip sync |
| GFPGAN weights | ~340 MB | Face quality enhancement |
| XTTS v2 | ~1.8 GB | Voice cloning (downloaded automatically in Section 4) |

In [ ]:
# Cell 2.1 — Download SadTalker model checkpoints from HuggingFace
# Uses snapshot_download to pull the full repo — robust against filename changes.

import os
from huggingface_hub import snapshot_download

CHECKPOINTS_DIR = "/content/SadTalker/checkpoints"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

# Check if already downloaded (look for a key file)
key_file = os.path.join(CHECKPOINTS_DIR, "mapping_00109-model.pth.tar")
if os.path.exists(key_file):
    print(f"SadTalker checkpoints already present in {CHECKPOINTS_DIR}")
else:
    print("Downloading SadTalker checkpoints (~900 MB)...")
    snapshot_download(
        repo_id="vinthony/SadTalker",
        local_dir=CHECKPOINTS_DIR,
        local_dir_use_symlinks=False,
        ignore_patterns=["*.md", ".gitattributes"],
    )
    print("Download complete.")

print(f"\nCheckpoint contents:")
!ls -lh {CHECKPOINTS_DIR}

In [ ]:
# Cell 2.2 — Download GFPGAN face enhancement weights
# GFPGAN restores and enhances facial details in the output video
# This significantly improves the quality of the final result

import os

GFPGAN_DIR = "/content/SadTalker/gfpgan/weights"
os.makedirs(GFPGAN_DIR, exist_ok=True)

gfpgan_files = [
    (
        "GFPGANv1.4.pth",
        "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    ),
    (
        "detection_Resnet50_Final.pth",
        "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
    ),
    (
        "parsing_parsenet.pth",
        "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
    ),
]

for filename, url in gfpgan_files:
    dest = os.path.join(GFPGAN_DIR, filename)
    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / 1024**2
        print(f"Already downloaded: {filename} ({size_mb:.1f} MB)")
    else:
        print(f"Downloading {filename}...")
        !wget -q --show-progress -O "{dest}" "{url}"
        size_mb = os.path.getsize(dest) / 1024**2
        print(f"Downloaded: {filename} ({size_mb:.1f} MB)")

print(f"\nAll GFPGAN weights ready in {GFPGAN_DIR}")
!ls -lh {GFPGAN_DIR}

---
## Section 3: Upload Reference Material

Upload your reference video (yourself speaking for ~1 minute).

The pipeline will automatically:
1. Extract a 30-second audio clip for voice cloning
2. Extract a face image for the talking head animation

> **Tip**: If your video is larger than 100 MB, trim it first or compress it. The voice clone only needs 30s of clean audio.

In [ ]:
# Cell 3.1 — Upload your reference video
# Supported formats: MP4, MOV, AVI, MKV

import os
from google.colab import files

WORKSPACE_DIR = "/content/workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

print("Click 'Choose Files' to upload your reference video.")
print("Recommended: 30s–2min of yourself speaking clearly.")
print()

uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Please upload a video file.")

# Get the uploaded filename
uploaded_filename = list(uploaded.keys())[0]
REF_VIDEO = os.path.join(WORKSPACE_DIR, uploaded_filename)

# Move to workspace if uploaded to root
if not os.path.exists(REF_VIDEO):
    import shutil
    shutil.move(uploaded_filename, REF_VIDEO)

file_size_mb = os.path.getsize(REF_VIDEO) / 1024**2
print(f"\nUploaded: {uploaded_filename}")
print(f"Size: {file_size_mb:.1f} MB")
print(f"Saved to: {REF_VIDEO}")

# Get video duration
import subprocess
result = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", REF_VIDEO],
    capture_output=True, text=True
)
if result.returncode == 0:
    duration = float(result.stdout.strip())
    print(f"Duration: {duration:.1f}s ({duration/60:.1f} min)")
    if duration < 10:
        print("WARNING: Video is very short. Voice clone quality may be poor. Aim for 30s+.")
    elif duration > 120:
        print("Note: Long video detected. Only the first 35s of audio will be used for voice reference.")

In [ ]:
# Cell 3.2 — Extract reference audio and face image from the video

import os
import subprocess

WORKSPACE_DIR = "/content/workspace"
REF_AUDIO = os.path.join(WORKSPACE_DIR, "reference_voice.wav")
REF_FACE = os.path.join(WORKSPACE_DIR, "reference_face.jpg")

# --- Extract 30 seconds of reference audio (starting at 5s to skip any intro)
print("Extracting reference audio (30s starting at 5s)...")
result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", REF_VIDEO,
        "-ss", "5",           # Start 5 seconds in
        "-t", "30",           # Take 30 seconds
        "-vn",               # No video
        "-acodec", "pcm_s16le",  # 16-bit PCM WAV
        "-ar", "22050",      # 22050 Hz sample rate (XTTS requirement)
        "-ac", "1",          # Mono
        REF_AUDIO,
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("Error extracting audio:")
    print(result.stderr[-1000:])
    raise RuntimeError("Audio extraction failed.")

audio_size_kb = os.path.getsize(REF_AUDIO) / 1024
print(f"Reference audio saved: {REF_AUDIO} ({audio_size_kb:.0f} KB)")

# --- Extract face image at 5 seconds into the video
print("\nExtracting reference face image (at 5s)...")
result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", REF_VIDEO,
        "-ss", "5",           # At 5 seconds
        "-frames:v", "1",    # Single frame
        "-q:v", "2",         # High quality JPEG
        REF_FACE,
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("Error extracting face image:")
    print(result.stderr[-1000:])
    raise RuntimeError("Face image extraction failed.")

face_size_kb = os.path.getsize(REF_FACE) / 1024
print(f"Reference face saved: {REF_FACE} ({face_size_kb:.0f} KB)")

print("\nExtraction complete.")
print(f"  Voice reference: {REF_AUDIO}")
print(f"  Face image:      {REF_FACE}")

In [ ]:
# Cell 3.3 — Preview reference material
# Verify the face looks good and the audio sounds like you before proceeding

import os
from IPython.display import display, Image, Audio

WORKSPACE_DIR = "/content/workspace"
REF_AUDIO = os.path.join(WORKSPACE_DIR, "reference_voice.wav")
REF_FACE  = os.path.join(WORKSPACE_DIR, "reference_face.jpg")

print("=" * 50)
print("Reference Face Image (will be animated by SadTalker):")
print("=" * 50)
display(Image(filename=REF_FACE, width=300))

print()
print("=" * 50)
print("Reference Voice Audio (30s — will be used for XTTS voice cloning):")
print("=" * 50)
display(Audio(filename=REF_AUDIO))

print()
print("Check:")
print("  [ ] Face is clearly visible, front-facing, well-lit")
print("  [ ] Audio sounds like you with minimal background noise")
print()
print("If the face or audio looks wrong, re-run Cell 3.2 with different -ss values")
print("to extract from a different point in the video.")

---
## Section 4: Generate Your Avatar Video

**Steps:**
1. Edit your script in Cell 4.1
2. Run Cell 4.2 to synthesize speech with your cloned voice
3. Run Cell 4.3 to generate the talking-head video
4. Run Cell 4.4 to download the result

> **Voice style**: Adjust  in Cell 4.2 (0 = calm/neutral, 0.5 = natural, 1 = expressive).
> Chatterbox TTS works best with English. For other languages, the voice clone may still work but accent naturalness varies.


In [ ]:
# Cell 4.1 — Edit your script here
# This is the text your AI avatar will say.
# Write naturally — punctuation helps with pacing.
# Aim for 50–300 words for best results.

SCRIPT = """
Hello, and welcome. My name is Bassem, and today I want to talk to you about
the future of artificial intelligence and how it is transforming the way we
work, communicate, and create.

Just a few years ago, the idea of cloning a voice or animating a face from
a single photo seemed like science fiction. Today, these technologies are
open source, freely available, and running right here on a free GPU in the cloud.

This pipeline you are watching was built entirely with open source tools:
Coqui XTTS for voice synthesis, and SadTalker for facial animation.
No paid APIs, no proprietary software — just the power of the open source community.

Thank you for watching, and I hope this inspires you to build something amazing.
""".strip()

# --- Statistics
words = len(SCRIPT.split())
wpm = 150  # average speaking rate
estimated_duration_min = words / wpm
estimated_duration_sec = estimated_duration_min * 60

print(f"Script loaded.")
print(f"  Word count:         {words} words")
print(f"  Estimated duration: {estimated_duration_sec:.0f}s (~{estimated_duration_min:.1f} min at {wpm} wpm)")
print()
print("Script preview (first 200 chars):")
print("-" * 40)
print(SCRIPT[:200] + ("..." if len(SCRIPT) > 200 else ""))

In [ ]:
# Cell 4.2 — Synthesize speech with Chatterbox TTS (your cloned voice)

import os, time, torch, numpy as np
from scipy.io import wavfile
from chatterbox.tts import ChatterboxTTS
from IPython.display import display, Audio

WORKSPACE_DIR = '/content/workspace'
REF_AUDIO     = os.path.join(WORKSPACE_DIR, 'reference_voice.wav')
OUTPUT_AUDIO  = os.path.join(WORKSPACE_DIR, 'generated_speech.wav')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

print('Loading Chatterbox TTS (~1 GB download on first run)...')
t0 = time.time()
model = ChatterboxTTS.from_pretrained(device=device)
print('Model loaded in', round(time.time() - t0, 1), 's')

print('Synthesizing speech...')
t1 = time.time()
wav = model.generate(
    SCRIPT,
    audio_prompt_path=REF_AUDIO,
    exaggeration=0.5,
)

# Save with scipy — no torchaudio needed
audio_np = wav.squeeze().cpu().numpy()
audio_int16 = (audio_np * 32767).astype(np.int16)
wavfile.write(OUTPUT_AUDIO, model.sr, audio_int16)

print('Done in', round(time.time() - t1, 1), 's')
print('Output:', OUTPUT_AUDIO)
display(Audio(filename=OUTPUT_AUDIO))


In [ ]:
# Cell 4.3 — Generate talking-head video with SadTalker
# SadTalker animates the reference face image to match the generated audio.
# --still:          reduces head motion for a more stable, professional look
# --preprocess full: processes the full face region (not just crop)
# --enhancer gfpgan: applies GFPGAN face restoration for higher quality
# --size 256:       output resolution 256x256 (faster; use 512 for higher quality)

import os
import time
import glob

WORKSPACE_DIR   = "/content/workspace"
SADTALKER_DIR   = "/content/SadTalker"
REF_FACE        = os.path.join(WORKSPACE_DIR, "reference_face.jpg")
OUTPUT_AUDIO    = os.path.join(WORKSPACE_DIR, "generated_speech.wav")
RESULT_DIR      = os.path.join(WORKSPACE_DIR, "sadtalker_output")

os.makedirs(RESULT_DIR, exist_ok=True)

print("Starting SadTalker inference...")
print(f"  Source image:  {REF_FACE}")
print(f"  Driven audio:  {OUTPUT_AUDIO}")
print(f"  Output dir:    {RESULT_DIR}")
print(f"  Resolution:    256x256")
print(f"  Face enhancer: GFPGAN")
print()
print("This takes ~3–5 minutes per minute of audio on a T4 GPU...")
print()

t0 = time.time()

!cd /content/SadTalker && python inference.py \
    --driven_audio "{OUTPUT_AUDIO}" \
    --source_image "{REF_FACE}" \
    --result_dir "{RESULT_DIR}" \
    --still \
    --preprocess full \
    --enhancer gfpgan \
    --size 256

elapsed = time.time() - t0
print()
print(f"SadTalker inference completed in {elapsed:.1f}s ({elapsed/60:.1f} min)")

# Find the output video
output_videos = glob.glob(os.path.join(RESULT_DIR, "**", "*.mp4"), recursive=True)

if output_videos:
    OUTPUT_VIDEO = sorted(output_videos, key=os.path.getmtime)[-1]  # Most recent
    size_mb = os.path.getsize(OUTPUT_VIDEO) / 1024**2
    print(f"Output video: {OUTPUT_VIDEO} ({size_mb:.1f} MB)")
else:
    print("WARNING: No output video found. Check the output above for errors.")
    print("Common issues:")
    print("  - Face not detected in reference image → try a different frame")
    print("  - Out of memory → restart runtime, try --size 256")

In [ ]:
# Cell 4.4 — Preview and download the final video

import os
import glob
import base64
from IPython.display import display, HTML
from google.colab import files

WORKSPACE_DIR = "/content/workspace"
RESULT_DIR = os.path.join(WORKSPACE_DIR, "sadtalker_output")

# Find the most recently generated MP4
output_videos = glob.glob(os.path.join(RESULT_DIR, "**", "*.mp4"), recursive=True)

if not output_videos:
    raise FileNotFoundError(
        "No output video found. Make sure Cell 4.3 completed successfully."
    )

OUTPUT_VIDEO = sorted(output_videos, key=os.path.getmtime)[-1]
size_mb = os.path.getsize(OUTPUT_VIDEO) / 1024**2
video_filename = os.path.basename(OUTPUT_VIDEO)

print(f"Output video: {OUTPUT_VIDEO}")
print(f"File size:    {size_mb:.1f} MB")
print()

# --- Display the video inline
print("Preview:")
with open(OUTPUT_VIDEO, "rb") as f:
    video_bytes = f.read()

video_b64 = base64.b64encode(video_bytes).decode()
video_html = f"""
<video width="480" controls autoplay loop>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
  Your browser does not support the video tag.
</video>
"""
display(HTML(video_html))

# --- Download to local machine
print()
print("Downloading video to your computer...")
files.download(OUTPUT_VIDEO)
print(f"Download initiated: {video_filename}")
print()
print("Done! Your AI avatar video has been generated and downloaded.")

---
## Tips & Troubleshooting

### If the voice doesn't sound like you:
- Use a longer, cleaner reference audio (30–60s with no background noise)
- Re-run Cell 3.2 with `-ss 10` or `-ss 0` to extract audio from a different segment
- Make sure the reference audio is mono and noise-free

### If the face is not detected:
- Re-run Cell 3.2 with a different `-ss` value to get a better frame
- Make sure your face is centered, well-lit, and front-facing in the reference video
- Avoid extreme angles or partial occlusion

### If you get an Out of Memory (OOM) error:
- Use `--size 256` (already set by default)
- Go to **Runtime → Restart runtime** and rerun from Section 4
- Shorten your script (fewer words = shorter audio = less memory)

### If SadTalker inference hangs:
- Check if the GPU is available (`!nvidia-smi`)
- Restart runtime and rerun from Cell 4.3 only (checkpoints are cached)

### For higher quality output:
- Change `--size 256` to `--size 512` in Cell 4.3 (needs more VRAM — use A100 or Colab Pro)
- Remove `--still` to allow natural head motion
- Use a 512x512 or larger reference face image

### Saving your work between sessions:
- Mount Google Drive: `from google.colab import drive; drive.mount('/content/drive')`
- Copy outputs: `!cp -r /content/workspace/sadtalker_output /content/drive/MyDrive/`
- Model checkpoints in `/content/SadTalker/checkpoints/` are lost when session ends — they will re-download automatically next time

---

### Changing the language
In Cell 4.2, change `language="en"` to any of:
`"en"`, `"es"`, `"fr"`, `"de"`, `"it"`, `"pt"`, `"pl"`, `"tr"`, `"ru"`, `"nl"`, `"cs"`, `"ar"`, `"zh-cn"`, `"ja"`, `"ko"`, `"hu"`, `"hi"`

XTTS v2 will generate speech in that language while preserving your voice characteristics.